In [ ]:
import pandas as pd
import re
from time import sleep
from bs4 import BeautifulSoup
import requests
from tqdm import tqdm
from datetime import datetime, timedelta
import calendar

def race_id_scrape(kaisai_nengetu) :
    
    ul1 = "https://keirin.kdreams.jp/gamboo/schedule/search/" + kaisai_nengetu
    res = requests.get(ul1)
    res.encoding = ['EUC-JP']
    soup = BeautifulSoup(res.text, 'html.parser')
    kaisai_sches = soup.find_all('td',attrs={'class':'kaisai'}) 
    nen = kaisai_nengetu[0:4]
    gatu = kaisai_nengetu[5:7]
    first_day = datetime.strptime(kaisai_nengetu + '/01', "%Y/%m/%d")
    last_day = datetime.strptime(kaisai_nengetu + '/' + str(calendar.monthrange(int(nen), int(gatu))[1]), "%Y/%m/%d")
    kaisai_lists = []
    kaisai_list = []


    for i in range(len(kaisai_sches)):
        if  kaisai_sches[i].find('a'):
            sche_ht = str(kaisai_sches[i].find('a'))
            sche_re = re.findall(r'\d+',sche_ht)
            kaisai_lists.append(sche_re[1])
        else :
            continue
    
    print(' {}年 {}月 の解析をしました。'.format(nen,gatu))
    
    # -----------------------------------------------------------------------
    # ------------------- 開催リストから開催日数分のリストを作る ----------------
    print('開催の日数をを調べます。')
    
    race_id_url = 'https://keirin.kdreams.jp/gamboo/keirin-kaisai/race-card/result/'


    for i in tqdm(range(len(kaisai_lists))): #len(kaisai_lists)
        
        
        # ------------------------URLの生成-------------------------------
        race_id_2 = kaisai_lists[i]
        race_id_1 = kaisai_lists[i] [0:10]
        race_id_3 = '01/'
        race_url = race_id_url + race_id_1 +"/"+race_id_2+"/"+race_id_3
        
        # -----------------------htmlの取得-------------------------------
        race_id_res = requests.get(race_url)
        race_id_res.encoding = ['EUC-JP']
        race_id_soup = BeautifulSoup(race_id_res.text, 'html.parser')
        
        # -----------------------開催日数の所得------------------------------
        day_len = len(race_id_soup.find_all('span',attrs={'class':'day'}))
        date_str = kaisai_lists[i]
        
    
        # -----------------------開催日数分のリストを追加---------------------
        for j in range(2,day_len+1):
            
            date_list =list(date_str)
            date_list[11:12] = str(j)
            date_list = ''.join(date_list)
            kaisai_lists.append(date_list) 
    
        sleep(1)        
    kaisai_lists.sort()    
    
    # -----------------------------月始から月末までを抜き取る ----------------
    for lis in kaisai_lists:
        r_date = datetime.strptime(lis[2:6]+'/'+lis[6:8]+'/'+lis[8:10],"%Y/%m/%d")
        ad_date = int(lis[10:12])-1
        r_date = r_date + timedelta(days=ad_date)
        
        if first_day <= r_date <= last_day :
            kaisai_list.append(lis)
    
    
    # -----------------------------------------------------------------------
    # ------------------------- すべてのレースIDを取得 ------------------------
    print('べてのレースIDをを調べます。')
    
    all_race_ids = []
    race_id_url = 'https://keirin.kdreams.jp/gamboo/keirin-kaisai/race-card/result/'
    
    for i in tqdm(range(len(kaisai_list))):#len(kaisai_lists)
        race_id_2 = kaisai_list[i]
        race_id_1 = kaisai_list[i] [0:10]
        race_id_3 = '01/'
        race_url = race_id_url + race_id_1 +"/"+race_id_2+"/"+race_id_3
        
        # -----------------------htmlの取得-------------------------------
        race_id_res = requests.get(race_url)
        race_id_res.encoding = ['EUC-JP']
        race_id_soup = BeautifulSoup(race_id_res.text, 'html.parser')
        
        sleep(1)
        
        #------------------------レース数の取得----------------------------
        try:
            if re.findall(r'\d+',race_id_soup.find('div',attrs={'class':'kaisai_race_data_nav'}).find_all('li')[-1].text)[0]:
                race_max_num = int(re.findall(r'\d+',race_id_soup.find('div',attrs={'class':'kaisai_race_data_nav'}).find_all('li')[-1].text)[0]) 
                #------------------------すべてのレースIDの取得--------------------
                for j in range(1,race_max_num+1,1):
                    all_race_url_num = str(j).zfill(2)
                    all_race_id = str(race_id_2+all_race_url_num)
                    all_race_ids.append(all_race_id)
                else:
                    continue
                
        except AttributeError:
            print(all_race_id,'開催が中止されています')
                    
            
        except IndexError:
            print('レースが中止されています')
            
    
    print(' {}年 {}月 の全 {} レースのIDを取得しました。'.format(nen,gatu,len(all_race_ids)))
    
    return all_race_ids

In [ ]:
# 取得した全レースIDを格納するマスターリスト
total_race_ids = []

# 2018年から2025年までループ (rangeは終了値の+1を指定します)
for year in range(2023, 2026):
    for month in range(1, 13):
        # 'YYYY/MM'の形式を作成（月は2桁でゼロ埋め）
        kaisai_nengetu = f"{year}/{month:02d}"

        kaisai_nengetu
        
        print(f"\n=== {kaisai_nengetu} の取得を開始 ===")
        
        # 関数を実行してその月のIDリストを取得
        monthly_ids = race_id_scrape(kaisai_nengetu)
        
        # 取得したリストをマスターリストに結合
        total_race_ids.extend(monthly_ids)
        
        # サーバー負荷への配慮として、月ごとの切り替わりでもインターバルを置く
        sleep(2)

print(f"\n全期間（2018〜2025年）の取得が完了しました。")
print(f"合計レースID数: {len(total_race_ids)}")

# 必要に応じてCSV等に保存しておくことをお勧めします
# import pandas as pd
# pd.Series(total_race_ids).to_csv('race_ids_2018_2025.csv', index=False, header=False)


=== 2018/01 の取得を開始 ===
 2018年 01月 の解析をしました。
開催の日数をを調べます。


100%|██████████| 62/62 [01:16<00:00,  1.24s/it]


べてのレースIDをを調べます。


100%|██████████| 181/181 [03:44<00:00,  1.24s/it]


 2018年 01月 の全 1881 レースのIDを取得しました。

=== 2018/02 の取得を開始 ===
 2018年 02月 の解析をしました。
開催の日数をを調べます。


...

べてのレースIDをを調べます。


100%|██████████| 219/219 [04:49<00:00,  1.32s/it]


 2023年 04月 の全 2259 レースのIDを取得しました。

=== 2023/05 の取得を開始 ===
 2023年 05月 の解析をしました。
開催の日数をを調べます。


  5%|▌         | 4/78 [00:27<08:21,  6.78s/it]


ReadTimeout: HTTPSConnectionPool(host='keirin.kdreams.jp', port=443): Read timed out. (read timeout=None)

In [ ]:
pd.to_pickle(total_race_ids,"total_race_ids.pkl")

In [4]:
total_race_ids

['1320180102010001',
 '1320180102010002',
 '1320180102010003',
 '1320180102010004',
 '1320180102010005',
 '1320180102010006',
 '1320180102010007',
 '1320180102010008',
 '1320180102010009',
 '1320180102010010',
 '1320180102010011',
 '1320180102020001',
 '1320180102020002',
 '1320180102020003',
 '1320180102020004',
 '1320180102020005',
 '1320180102020006',
 '1320180102020007',
 '1320180102020008',
 '1320180102020009',
 '1320180102020010',
 '1320180102020011',
 '1320180102030001',
 '1320180102030002',
 '1320180102030003',
 '1320180102030004',
 '1320180102030005',
 '1320180102030006',
 '1320180102030007',
 '1320180102030008',
 '1320180102030009',
 '1320180102030010',
 '1320180102030011',
 '2220180114010001',
 '2220180114010002',
 '2220180114010003',
 '2220180114010004',
 '2220180114010005',
 '2220180114010006',
 '2220180114010007',
 '2220180114010008',
 '2220180114010009',
 '2220180114010010',
 '2220180114010011',
 '2220180114020001',
 '2220180114020002',
 '2220180114020003',
 '22201801140

In [6]:
import pickle
import pandas as pd

# --- 1. Pickle形式で保存 ---
# Python専用のバイナリ形式で、後で読み込むのが一番速く確実です
with open('scraped_data/pickle/total_race_ids_backup.pkl', 'wb') as f:
    pickle.dump(total_race_ids, f)
print("Pickleでの保存が完了しました。")

# --- 2. CSV形式で保存 ---
# 他のソフト（Excelなど）で確認したい場合や汎用性を持たせたい場合に便利です
pd.Series(total_race_ids).to_csv('scraped_data/csv/total_race_ids_backup.csv', index=False, header=False)
print("CSVでの保存が完了しました。")

# （確認用）現在取得できている件数を表示
print(f"現在保存されたレースID数: {len(total_race_ids)}")

Pickleでの保存が完了しました。
CSVでの保存が完了しました。
現在保存されたレースID数: 129401


In [7]:
# 【安全対策1】CSVで上書き保存
pd.Series(total_race_ids).to_csv('scraped_data/pickle/total_race_ids_progress.csv', index=False, header=False)

# 【安全対策2】Pickleで上書き保存（追加）
with open('scraped_data/csv/total_race_ids_progress.pkl', 'wb') as f:
    pickle.dump(total_race_ids, f)

In [ ]:
# TODO: パソ子がどんなデータを取得しているか調査
# TODO: パソ子が取得しているデータのスクレイピングコードを作成
# TODO: CSV形式でテーブルの作成とpickleでそのHTMLを保存する